In [0]:
%python
display(dbutils.fs.ls("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"))

path,name,size,modificationTime
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,Fact_Sales_1.csv,299478,1785686941000


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS delta_forge_catalog;
CREATE SCHEMA IF NOT EXISTS delta_forge_catalog.legacy_hms_db;

In [0]:
%python

from pyspark.sql.functions import col

raw_path = "/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"
checkpoint_path = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"

df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", schema_location)
  .option("header", "true")
  .load(raw_path)
  .withColumn("file_name", col("_metadata.file_path"))
  .withColumn("file_arrival_time", col("_metadata.file_modification_time"))
)

(df.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
)

In [0]:
%python
display(spark.sql("SELECT * FROM delta_forge_catalog.legacy_hms_db.bronze_transactions LIMIT 10"))

transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price,_rescued_data,file_name,file_arrival_time
1,04-05-2021 02:00,P0494,4,visa,4.04159E+15,F,17.33,2,18.29,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,2026-08-02T16:09:01.000Z
2,04-05-2021 03:04,P0221,5,visa,4.0416E+15,F,0.59,1,1.49,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,2026-08-02T16:09:01.000Z
3,04-05-2021 03:56,P0625,5,visa,4.04159E+15,F,5.15,3,5.89,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,2026-08-02T16:09:01.000Z
4,04-05-2021 05:20,P0431,8,mastercard,5.10875E+15,F,10.67,2,11.59,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,2026-08-02T16:09:01.000Z
5,04-05-2021 05:45,P0058,5,mastercard,5.10875E+15,T,11.38,2,12.39,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,2026-08-02T16:09:01.000Z
6,04-05-2021 06:58,P0385,6,americanexpress,3.74289E+14,F,13.22,1,14.69,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,2026-08-02T16:09:01.000Z
7,04-05-2021 07:03,P0575,4,visa,4.0416E+12,F,2.81,1,3.99,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,2026-08-02T16:09:01.000Z
8,04-05-2021 07:45,P0187,5,americanexpress,3.74283E+14,F,4.17,1,4.89,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,2026-08-02T16:09:01.000Z
9,04-05-2021 09:58,P0074,7,mastercard,5.10875E+15,F,18.11,1,19.79,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,2026-08-02T16:09:01.000Z
10,04-05-2021 15:51,P0456,5,mastercard,5.04837E+15,T,18.35,2,20.19,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,2026-08-02T16:09:01.000Z


In [0]:
%sql
DESCRIBE delta_forge_catalog.legacy_hms_db.bronze_transactions;

col_name,data_type,comment
transaction_id,string,null
transactional_date,string,null
product_id,string,null
customer_id,string,null
payment,string,null
credit_card,string,null
loyalty_card,string,null
cost,string,null
quantity,string,null
price,string,null


In [0]:
%sql
CREATE TABLE IF NOT EXISTS delta_forge_catalog.legacy_hms_db.silver_transactions (
    transaction_id STRING,
    transactional_date STRING,
    product_id STRING,
    customer_id STRING,
    payment STRING,
    credit_card STRING,
    loyalty_card STRING,
    cost STRING,
    quantity STRING,
    price STRING,
    amount DOUBLE,
    file_name STRING,
    file_arrival_time TIMESTAMP,
    processed_time TIMESTAMP,
    processed_flag INT
) USING DELTA;

In [0]:
%sql
DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.silver_transactions;

CREATE TABLE delta_forge_catalog.legacy_hms_db.silver_transactions (
    transaction_id STRING,
    transactional_date STRING,
    product_id STRING,
    customer_id STRING,
    payment STRING,
    credit_card STRING,
    loyalty_card STRING,
    cost DOUBLE,
    quantity INT,
    price DOUBLE,
    amount DOUBLE,
    file_name STRING,
    file_arrival_time TIMESTAMP,
    processed_time TIMESTAMP,
    processed_flag INT
) USING DELTA;

In [0]:
%python

from pyspark.sql.functions import current_timestamp, lit, col

def process_silver_microbatch(microBatchDF, batchId):
    cleaned_df = (microBatchDF
        .dropna(subset=["transaction_id", "price", "quantity"])
        .dropDuplicates(["transaction_id"])
        .withColumn("cost", col("cost").cast("double"))
        .withColumn("quantity", col("quantity").cast("int"))
        .withColumn("price", col("price").cast("double"))
        .withColumn("amount", col("price") * col("quantity"))
        .withColumn("processed_time", current_timestamp())
        .withColumn("processed_flag", lit(1)))

    cleaned_df.createOrReplaceTempView("silver_updates")

    cleaned_df.sparkSession.sql("""
        MERGE INTO delta_forge_catalog.legacy_hms_db.silver_transactions AS target
        USING silver_updates AS source
        ON target.transaction_id = source.transaction_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
%python
silver_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/silver/"

bronze_stream = spark.readStream.table("delta_forge_catalog.legacy_hms_db.bronze_transactions")

(bronze_stream.writeStream
    .foreachBatch(process_silver_microbatch)
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .start()
)

In [0]:
%sql
SELECT transaction_id, cost, quantity, price, amount, processed_flag
FROM delta_forge_catalog.legacy_hms_db.silver_transactions
LIMIT 10;

transaction_id,cost,quantity,price,amount,processed_flag
56,11.76,3,13.09,39.269999999999996,1
73,2.95,5,4.09,20.45,1
110,1.6,4,1.79,7.16,1
120,15.78,1,17.59,17.59,1
146,17.42,1,18.49,18.49,1
149,19.79,1,21.39,21.39,1
155,16.85,1,18.19,18.19,1
156,5.35,1,6.29,6.29,1
167,1.03,1,1.59,1.59,1
168,15.32,2,17.09,34.18,1


In [0]:
%sql
SELECT COUNT(*) FROM delta_forge_catalog.legacy_hms_db.silver_transactions;

COUNT(*)
4410


In [0]:
%sql
SELECT COUNT(*) FROM delta_forge_catalog.legacy_hms_db.bronze_transactions;

COUNT(*)
4410


In [0]:
%python
from pyspark.sql.functions import col

raw_path = "/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"
checkpoint_path = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"

df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", schema_location)
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  .option("cloudFiles.rescuedDataColumn", "_rescued_data")
  .option("header", "true")
  .load(raw_path)
  .withColumn("file_name", col("_metadata.file_path"))
  .withColumn("file_arrival_time", col("_metadata.file_modification_time"))
)

(df.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
)

In [0]:
%python
import pandas as pd
import random

# Read the original file from your volume
df = pd.read_csv("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2.csv")

# Add the new column with sample values
df["channel"] = [random.choice(["online", "in_store"]) for _ in range(len(df))]

# Save as a new file directly into the volume
df.to_csv("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv", index=False)

print(f"Rows written: {len(df)}")
print(df.head())

Rows written: 40
   transaction_id transactional_date product_id  ...  quantity  price   channel
0            4411   05-01-2022 01:02      P0305  ...         3   9.99    online
1            4412   05-01-2022 04:00      P0242  ...         2   1.79  in_store
2            4413   05-01-2022 05:49      P0529  ...         2  18.79    online
3            4414   05-01-2022 06:58      P0336  ...         4   8.89  in_store
4            4415   05-01-2022 08:47      P0399  ...         2  11.29  in_store

[5 rows x 11 columns]


In [0]:
%python
from pyspark.sql.functions import col

raw_path = "/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"
checkpoint_path = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"

df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", schema_location)
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  .option("cloudFiles.rescuedDataColumn", "_rescued_data")
  .option("header", "true")
  .load(raw_path)
  .withColumn("file_name", col("_metadata.file_path"))
  .withColumn("file_arrival_time", col("_metadata.file_modification_time"))
)

(df.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
)

In [0]:
%python
from pyspark.sql.functions import col

raw_path = "/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"
checkpoint_path = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"

df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", schema_location)
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  .option("cloudFiles.rescuedDataColumn", "_rescued_data")
  .option("header", "true")
  .load(raw_path)
  .withColumn("file_name", col("_metadata.file_path"))
  .withColumn("file_arrival_time", col("_metadata.file_modification_time"))
)

(df.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
)

In [0]:
%sql
DESCRIBE delta_forge_catalog.legacy_hms_db.bronze_transactions;

col_name,data_type,comment
transaction_id,string,null
transactional_date,string,null
product_id,string,null
customer_id,string,null
payment,string,null
credit_card,string,null
loyalty_card,string,null
cost,string,null
quantity,string,null
price,string,null


In [0]:
%sql
SELECT transaction_id, channel, file_name
FROM delta_forge_catalog.legacy_hms_db.bronze_transactions
WHERE file_name LIKE '%channel%'
LIMIT 10;

transaction_id,channel,file_name
4411,online,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv
4412,in_store,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv
4413,online,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv
4414,in_store,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv
4415,in_store,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv
4416,online,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv
4417,online,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv
4418,online,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv
4419,online,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv
4420,in_store,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv


In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.bronze_transactions
WHERE transaction_id = 1;

transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price,_rescued_data,file_name,file_arrival_time,channel
1,04-05-2021 02:00,P0494,4,visa,4.04159E+15,F,17.33,2,18.29,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,2026-08-02T16:09:01.000Z,null


In [0]:
%python

import pandas as pd

# Manually build one row simulating a late-arriving correction for transaction_id = 1
late_data = {
    "transaction_id": ["1"],
    "transactional_date": ["01-05-2021 09:00"],   # an earlier/different date than the original
    "product_id": ["P0494"],                        # keep same product_id as original if you noted it, or adjust
    "customer_id": ["4"],
    "payment": ["visa"],
    "credit_card": ["4041590000000000"],
    "loyalty_card": ["F"],
    "cost": ["15.99"],      # deliberately different from the original value
    "quantity": ["2"],
    "price": ["16.99"]      # deliberately different from the original value
}

df = pd.DataFrame(late_data)

df.to_csv("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/late_arrival_test.csv", index=False)

print("File written:")
print(df)

File written:
  transaction_id transactional_date product_id  ...   cost quantity  price
0              1   01-05-2021 09:00      P0494  ...  15.99        2  16.99

[1 rows x 10 columns]


In [0]:
%python
display(dbutils.fs.ls("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"))

path,name,size,modificationTime
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,Fact_Sales_1.csv,299478,1785686941000
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2.csv,Fact_Sales_2.csv,2795,1785735291000
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv,Fact_Sales_2_with_channel.csv,3321,1785735326000
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/late_arrival_test.csv,late_arrival_test.csv,175,1785736733000


In [0]:
%sql
SELECT COUNT(*) AS row_count_before FROM delta_forge_catalog.legacy_hms_db.silver_transactions;

row_count_before
4410


In [0]:
%python
from pyspark.sql.functions import col

raw_path = "/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"
checkpoint_path = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"

df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", schema_location)
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  .option("cloudFiles.rescuedDataColumn", "_rescued_data")
  .option("header", "true")
  .load(raw_path)
  .withColumn("file_name", col("_metadata.file_path"))
  .withColumn("file_arrival_time", col("_metadata.file_modification_time"))
)

(df.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
)

In [0]:
%python
from pyspark.sql.functions import current_timestamp, lit, col

def process_silver_microbatch(microBatchDF, batchId):
    cleaned_df = (microBatchDF
        .dropna(subset=["transaction_id", "price", "quantity"])
        .dropDuplicates(["transaction_id"])
        .withColumn("cost", col("cost").cast("double"))
        .withColumn("quantity", col("quantity").cast("int"))
        .withColumn("price", col("price").cast("double"))
        .withColumn("amount", col("price") * col("quantity"))
        .withColumn("processed_time", current_timestamp())
        .withColumn("processed_flag", lit(1)))

    cleaned_df.createOrReplaceTempView("silver_updates")

    cleaned_df.sparkSession.sql("""
        MERGE INTO delta_forge_catalog.legacy_hms_db.silver_transactions AS target
        USING silver_updates AS source
        ON target.transaction_id = source.transaction_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
%python
silver_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/silver/"

bronze_stream = spark.readStream.table("delta_forge_catalog.legacy_hms_db.bronze_transactions")

(bronze_stream.writeStream
    .foreachBatch(process_silver_microbatch)
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .start()
)

In [0]:
%sql
SELECT COUNT(*) AS row_count_after FROM delta_forge_catalog.legacy_hms_db.silver_transactions;

row_count_after
4450


In [0]:
%sql
SELECT COUNT(*) FROM delta_forge_catalog.legacy_hms_db.bronze_transactions;

COUNT(*)
4491


In [0]:
%sql
SELECT file_name, COUNT(*) AS row_count
FROM delta_forge_catalog.legacy_hms_db.bronze_transactions
GROUP BY file_name
ORDER BY file_name;

file_name,row_count
/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,4410
/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2.csv,40
/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv,40
/Volumes/delta-forge-catalog/bronze/raw_landing_vol/late_arrival_test.csv,1


In [0]:
%sql
SELECT transaction_id, cost, price, quantity, processed_time
FROM delta_forge_catalog.legacy_hms_db.silver_transactions
WHERE transaction_id = 1;

transaction_id,cost,price,quantity,processed_time
1,15.99,16.99,2,2026-08-03T06:10:11.822Z


In [0]:
%sql
SELECT file_name, _rescued_data
FROM delta_forge_catalog.legacy_hms_db.bronze_transactions
WHERE _rescued_data IS NOT NULL
LIMIT 10;

file_name,_rescued_data


In [0]:
%python
raw_path_local = "/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"
files = dbutils.fs.ls(raw_path_local)
zero_byte_files = [f.path for f in files if f.size == 0]

if zero_byte_files:
    print(f"WARNING: found {len(zero_byte_files)} zero-byte file(s): {zero_byte_files}")
else:
    print("No zero-byte files found — safe to proceed.")

No zero-byte files found — safe to proceed.


In [0]:
%sql
OPTIMIZE delta_forge_catalog.legacy_hms_db.bronze_transactions;
OPTIMIZE delta_forge_catalog.legacy_hms_db.silver_transactions;


path,metrics
,"List(1, 2, List(67900, 67900, 67900.0, 1, 67900), List(6786, 66308, 36547.0, 2, 73094), 0, null, null, 0, 1, 2, 0, true, 0, 0, 1785737966639, 1785737968321, 8, 1, null, List(1, 1), null, 15, 15, 408, 0, null, null)"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS delta_forge_catalog.legacy_hms_db.gold_kpis (
    window_start TIMESTAMP,
    window_end TIMESTAMP,
    total_records_processed BIGINT,
    total_transaction_value DOUBLE,
    new_files_ingested BIGINT,
    duplicate_records_eliminated BIGINT,
    latest_data_freshness TIMESTAMP,
    freshness_lag_minutes DOUBLE,
    kpi_computed_at TIMESTAMP
) USING DELTA;

In [0]:

from pyspark.sql.functions import (
    window, count, sum as _sum, max as _max, countDistinct,
    current_timestamp, unix_timestamp, col
)

gold_df = (spark.readStream.table("delta_forge_catalog.legacy_hms_db.silver_transactions")
    .groupBy(window("processed_time", "1 hour"))
    .agg(
        count("transaction_id").alias("total_records_processed"),
        _sum("cost").alias("total_transaction_value"),
        countDistinct("file_name").alias("new_files_ingested"),
        _max("processed_time").alias("latest_data_freshness")
    )
    .withColumn("window_start", col("window.start"))
    .withColumn("window_end", col("window.end"))
    .withColumn("kpi_computed_at", current_timestamp())
    .withColumn(
        "freshness_lag_minutes",
        (unix_timestamp(current_timestamp()) - unix_timestamp(col("latest_data_freshness"))) / 60
    )
    .drop("window")
)

In [0]:
from pyspark.sql.streaming import StreamingQueryListener

class GoldBatchMetricsListener(StreamingQueryListener):
    def onQueryStarted(self, event):
        print(f"Query started: {event.id}")

    def onQueryProgress(self, event):
        progress = event.progress
        batch_id = progress.batchId
        duration_ms = progress.durationMs.get("triggerExecution", None)
        input_rows = progress.numInputRows
        print(f"Batch {batch_id}: {input_rows} rows processed in {duration_ms} ms")
        if duration_ms is not None:
            spark.sql(f"""
                INSERT INTO delta_forge_catalog.legacy_hms_db.gold_batch_metrics
                VALUES ({batch_id}, {input_rows}, {duration_ms}, current_timestamp())
            """)

    def onQueryTerminated(self, event):
        print(f"Query terminated: {event.id}")

listener = GoldBatchMetricsListener()
spark.streams.addListener(listener)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delta_forge_catalog.legacy_hms_db.gold_batch_metrics (
    batch_id BIGINT,
    input_rows BIGINT,
    duration_ms BIGINT,
    recorded_at TIMESTAMP
) USING DELTA;

In [0]:
bronze_count = spark.sql("SELECT COUNT(*) AS c FROM delta_forge_catalog.legacy_hms_db.bronze_transactions").collect()[0]["c"]
silver_count = spark.sql("SELECT COUNT(*) AS c FROM delta_forge_catalog.legacy_hms_db.silver_transactions").collect()[0]["c"]
duplicates_eliminated = bronze_count - silver_count

print(f"Bronze: {bronze_count}, Silver: {silver_count}, Duplicates eliminated: {duplicates_eliminated}")

spark.sql(f"""
    UPDATE delta_forge_catalog.legacy_hms_db.gold_kpis
    SET duplicate_records_eliminated = {duplicates_eliminated}
    WHERE kpi_computed_at = (SELECT MAX(kpi_computed_at) FROM delta_forge_catalog.legacy_hms_db.gold_kpis)
""")

Bronze: 4491, Silver: 4450, Duplicates eliminated: 41


DataFrame[num_affected_rows: bigint]

In [0]:
from pyspark.sql.functions import (
    window, count, sum as _sum, max as _max, countDistinct,
    current_timestamp, unix_timestamp, col
)

gold_df = (spark.readStream.table("delta_forge_catalog.legacy_hms_db.silver_transactions")
    .withWatermark("processed_time", "10 minutes")
    .groupBy(window("processed_time", "1 hour"))
    .agg(
        count("transaction_id").alias("total_records_processed"),
        _sum("cost").alias("total_transaction_value"),
        countDistinct("file_name").alias("new_files_ingested"),
        _max("processed_time").alias("latest_data_freshness")
    )
    .withColumn("window_start", col("window.start"))
    .withColumn("window_end", col("window.end"))
    .withColumn("kpi_computed_at", current_timestamp())
    .withColumn(
        "freshness_lag_minutes",
        (unix_timestamp(current_timestamp()) - unix_timestamp(col("latest_data_freshness"))) / 60
    )
    .drop("window")
)

In [0]:
gold_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold/"

(gold_df.writeStream
    .format("delta")
    .option("checkpointLocation", gold_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .table("delta_forge_catalog.legacy_hms_db.gold_kpis")
)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8692544039020506>, line 3
      1 gold_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold/"
----> 3 (gold_df.writeStream
      4     .format("delta")
      5     .option("checkpointLocation", gold_checkpoint)
      6     .outputMode("append")
      7     .trigger(availableNow=True)
      8     .table("delta_forge_catalog.legacy_hms_db.gold_kpis")
      9 )

NameError: name 'gold_df' is not defined

In [0]:
from pyspark.sql.functions import (
    window, count, sum as _sum, max as _max, approx_count_distinct,
    current_timestamp, unix_timestamp, col
)

gold_df = (spark.readStream.table("delta_forge_catalog.legacy_hms_db.silver_transactions")
    .withWatermark("processed_time", "10 minutes")
    .groupBy(window("processed_time", "1 hour"))
    .agg(
        count("transaction_id").alias("total_records_processed"),
        _sum("cost").alias("total_transaction_value"),
        approx_count_distinct("file_name").alias("new_files_ingested"),
        _max("processed_time").alias("latest_data_freshness")
    )
    .withColumn("window_start", col("window.start"))
    .withColumn("window_end", col("window.end"))
    .withColumn("kpi_computed_at", current_timestamp())
    .withColumn(
        "freshness_lag_minutes",
        (unix_timestamp(current_timestamp()) - unix_timestamp(col("latest_data_freshness"))) / 60
    )
    .drop("window")
)

In [0]:
gold_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold/"

(gold_df.writeStream
    .format("delta")
    .option("checkpointLocation", gold_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .table("delta_forge_catalog.legacy_hms_db.gold_kpis")
)

Query started: 0462c7fb-95d5-44d9-a72f-5a9c3df05c8f


In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.gold_kpis
ORDER BY window_start DESC;

window_start,window_end,total_records_processed,total_transaction_value,new_files_ingested,duplicate_records_eliminated,latest_data_freshness,freshness_lag_minutes,kpi_computed_at
2026-08-03T05:00:00.000Z,2026-08-03T06:00:00.000Z,4409,44760.28000000009,1,null,2026-08-03T05:20:03.311Z,75.83333333333333,2026-08-03T06:35:53.185Z


In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.gold_batch_metrics
ORDER BY recorded_at DESC;

batch_id,input_rows,duration_ms,recorded_at
1,0,5584,2026-08-03T06:35:59.117Z
0,4450,13972,2026-08-03T06:35:53.121Z


In [0]:
from pyspark.sql.functions import (
    window, count, sum as _sum, max as _max, approx_count_distinct,
    current_timestamp, unix_timestamp, col
)

gold_df_complete = (spark.readStream.table("delta_forge_catalog.legacy_hms_db.silver_transactions")
    .groupBy(window("processed_time", "1 hour"))
    .agg(
        count("transaction_id").alias("total_records_processed"),
        _sum("cost").alias("total_transaction_value"),
        approx_count_distinct("file_name").alias("new_files_ingested"),
        _max("processed_time").alias("latest_data_freshness")
    )
    .withColumn("window_start", col("window.start"))
    .withColumn("window_end", col("window.end"))
    .withColumn("kpi_computed_at", current_timestamp())
    .withColumn(
        "freshness_lag_minutes",
        (unix_timestamp(current_timestamp()) - unix_timestamp(col("latest_data_freshness"))) / 60
    )
    .drop("window")
)

In [0]:
gold_checkpoint_complete = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold_complete/"

(gold_df_complete.writeStream
    .format("delta")
    .option("checkpointLocation", gold_checkpoint_complete)
    .outputMode("complete")
    .trigger(availableNow=True)
    .table("delta_forge_catalog.legacy_hms_db.gold_kpis_complete")
)

Query started: 5efcdff9-2b99-457f-8322-72ea1dd4a9ea


In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.gold_kpis_complete
ORDER BY window_start;

total_records_processed,total_transaction_value,new_files_ingested,latest_data_freshness,window_start,window_end,kpi_computed_at,freshness_lag_minutes
4409,44760.28000000009,1,2026-08-03T05:20:03.311Z,2026-08-03T05:00:00.000Z,2026-08-03T06:00:00.000Z,2026-08-03T06:40:26.374Z,80.38333333333334
41,418.08000000000015,2,2026-08-03T06:10:11.822Z,2026-08-03T06:00:00.000Z,2026-08-03T07:00:00.000Z,2026-08-03T06:40:26.374Z,30.25


In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
dbutils.widgets.text("base_path", "abfss://delta-forge@deltaforge.dfs.core.windows.net/", "Base Path")

environment = dbutils.widgets.get("environment")
base_path = dbutils.widgets.get("base_path")

In [0]:
raw_path = f"{base_path}raw_landing/"
schema_location = f"{base_path}checkpoints/schema/"
checkpoint_path = f"{base_path}checkpoints/bronze/"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delta_forge_catalog.legacy_hms_db.pipeline_audit_log (
    run_id STRING,
    layer STRING,
    file_name STRING,
    row_count BIGINT,
    status STRING,
    error_message STRING,
    run_timestamp TIMESTAMP
) USING DELTA;

In [0]:
import uuid
run_id = str(uuid.uuid4())
print(f"This run's ID: {run_id}")

This run's ID: a8122820-4cce-4db1-90f6-adfc6020e0f9


In [0]:
from pyspark.sql.functions import current_timestamp, lit

def log_audit(run_id, layer, file_name, row_count, status, error_message=None):
    spark.sql(f"""
        INSERT INTO delta_forge_catalog.legacy_hms_db.pipeline_audit_log
        VALUES (
            '{run_id}',
            '{layer}',
            {"'" + file_name + "'" if file_name else "NULL"},
            {row_count if row_count is not None else "NULL"},
            '{status}',
            {"'" + error_message.replace("'", "''") + "'" if error_message else "NULL"},
            current_timestamp()
        )
    """)

In [0]:
raw_path = "/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"
checkpoint_path = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"

In [0]:
import uuid
run_id = str(uuid.uuid4())
print(f"This run's ID: {run_id}")

This run's ID: 4076f567-0f45-47a4-9684-97a1aecc4fcb


In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

def log_audit(run_id, layer, file_name, row_count, status, error_message=None):
    spark.sql(f"""
        INSERT INTO delta_forge_catalog.legacy_hms_db.pipeline_audit_log
        VALUES (
            '{run_id}',
            '{layer}',
            {"'" + file_name + "'" if file_name else "NULL"},
            {row_count if row_count is not None else "NULL"},
            '{status}',
            {"'" + error_message.replace("'", "''") + "'" if error_message else "NULL"},
            current_timestamp()
        )
    """)

In [0]:
log_audit(run_id, "bronze", None, None, "STARTED")

try:
    df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schema_location)
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
      .option("cloudFiles.rescuedDataColumn", "_rescued_data")
      .option("header", "true")
      .load(raw_path)
      .withColumn("file_name", col("_metadata.file_path"))
      .withColumn("file_arrival_time", col("_metadata.file_modification_time"))
    )

    query = (df.writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .option("mergeSchema", "true")
      .trigger(availableNow=True)
      .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
    )
    query.awaitTermination()

    row_count = spark.sql("SELECT COUNT(*) AS c FROM delta_forge_catalog.legacy_hms_db.bronze_transactions").collect()[0]["c"]
    log_audit(run_id, "bronze", None, row_count, "SUCCESS")

except Exception as e:
    log_audit(run_id, "bronze", None, None, "FAILED", str(e))
    raise

In [0]:
def process_silver_microbatch(microBatchDF, batchId):
    try:
        cleaned_df = (microBatchDF
            .dropna(subset=["transaction_id", "price", "quantity"])
            .dropDuplicates(["transaction_id"])
            .withColumn("cost", col("cost").cast("double"))
            .withColumn("quantity", col("quantity").cast("int"))
            .withColumn("price", col("price").cast("double"))
            .withColumn("amount", col("price") * col("quantity"))
            .withColumn("processed_time", current_timestamp())
            .withColumn("processed_flag", lit(1)))

        cleaned_df.createOrReplaceTempView("silver_updates")

        cleaned_df.sparkSession.sql("""
            MERGE INTO delta_forge_catalog.legacy_hms_db.silver_transactions AS target
            USING silver_updates AS source
            ON target.transaction_id = source.transaction_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

        batch_row_count = cleaned_df.count()
        log_audit(run_id, "silver", f"batch_{batchId}", batch_row_count, "SUCCESS")

    except Exception as e:
        log_audit(run_id, "silver", f"batch_{batchId}", None, "FAILED", str(e))
        raise

In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.pipeline_audit_log
ORDER BY run_timestamp DESC;

run_id,layer,file_name,row_count,status,error_message,run_timestamp
4076f567-0f45-47a4-9684-97a1aecc4fcb,bronze,null,4491,SUCCESS,null,2026-08-06T06:52:37.798Z
4076f567-0f45-47a4-9684-97a1aecc4fcb,bronze,null,null,STARTED,null,2026-08-06T06:52:28.587Z
a8122820-4cce-4db1-90f6-adfc6020e0f9,bronze,null,null,FAILED,name schema_location is not defined,2026-08-06T06:48:52.726Z
a8122820-4cce-4db1-90f6-adfc6020e0f9,bronze,null,null,STARTED,null,2026-08-06T06:48:41.482Z


In [0]:
# TEMPORARY - induces a failure to test error handling
raw_path_test = "/Volumes/delta-forge-catalog/bronze/nonexistent_folder/"

In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.pipeline_audit_log
WHERE status = 'FAILED'
ORDER BY run_timestamp DESC;

run_id,layer,file_name,row_count,status,error_message,run_timestamp
a8122820-4cce-4db1-90f6-adfc6020e0f9,bronze,null,null,FAILED,name schema_location is not defined,2026-08-06T06:48:52.726Z


In [0]:
raw_path = "/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"
checkpoint_path = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"

In [0]:
log_audit(run_id, "bronze", None, None, "STARTED")

try:
    df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schema_location)
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
      .option("cloudFiles.rescuedDataColumn", "_rescued_data")
      .option("header", "true")
      .load(raw_path)
      .withColumn("file_name", col("_metadata.file_path"))
      .withColumn("file_arrival_time", col("_metadata.file_modification_time"))
    )

    query = (df.writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .option("mergeSchema", "true")
      .trigger(availableNow=True)
      .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
    )
    query.awaitTermination()

    row_count = spark.sql("SELECT COUNT(*) AS c FROM delta_forge_catalog.legacy_hms_db.bronze_transactions").collect()[0]["c"]
    log_audit(run_id, "bronze", None, row_count, "SUCCESS")

except Exception as e:
    log_audit(run_id, "bronze", None, None, "FAILED", str(e))
    raise

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

def process_silver_microbatch(microBatchDF, batchId):
    try:
        cleaned_df = (microBatchDF
            .dropna(subset=["transaction_id", "price", "quantity"])
            .dropDuplicates(["transaction_id"])
            .withColumn("cost", col("cost").cast("double"))
            .withColumn("quantity", col("quantity").cast("int"))
            .withColumn("price", col("price").cast("double"))
            .withColumn("amount", col("price") * col("quantity"))
            .withColumn("processed_time", current_timestamp())
            .withColumn("processed_flag", lit(1)))

        cleaned_df.createOrReplaceTempView("silver_updates")

        cleaned_df.sparkSession.sql("""
            MERGE INTO delta_forge_catalog.legacy_hms_db.silver_transactions AS target
            USING silver_updates AS source
            ON target.transaction_id = source.transaction_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

        batch_row_count = cleaned_df.count()
        log_audit(run_id, "silver", f"batch_{batchId}", batch_row_count, "SUCCESS")

    except Exception as e:
        log_audit(run_id, "silver", f"batch_{batchId}", None, "FAILED", str(e))
        raise

In [0]:
silver_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/silver/"

bronze_stream = spark.readStream.table("delta_forge_catalog.legacy_hms_db.bronze_transactions")

(bronze_stream.writeStream
    .foreachBatch(process_silver_microbatch)
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .start()
)

In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.pipeline_audit_log
ORDER BY run_timestamp DESC
LIMIT 10;

run_id,layer,file_name,row_count,status,error_message,run_timestamp
4076f567-0f45-47a4-9684-97a1aecc4fcb,silver,batch_2,0,SUCCESS,null,2026-08-06T07:05:22.611Z
4076f567-0f45-47a4-9684-97a1aecc4fcb,bronze,null,4491,SUCCESS,null,2026-08-06T06:58:22.772Z
4076f567-0f45-47a4-9684-97a1aecc4fcb,bronze,null,null,STARTED,null,2026-08-06T06:58:18.367Z
4076f567-0f45-47a4-9684-97a1aecc4fcb,bronze,null,4491,SUCCESS,null,2026-08-06T06:52:37.798Z
4076f567-0f45-47a4-9684-97a1aecc4fcb,bronze,null,null,STARTED,null,2026-08-06T06:52:28.587Z
a8122820-4cce-4db1-90f6-adfc6020e0f9,bronze,null,null,FAILED,name schema_location is not defined,2026-08-06T06:48:52.726Z
a8122820-4cce-4db1-90f6-adfc6020e0f9,bronze,null,null,STARTED,null,2026-08-06T06:48:41.482Z


In [0]:
%sql
CREATE TABLE IF NOT EXISTS delta_forge_catalog.legacy_hms_db.silver_quarantine (
    transaction_id STRING,
    transactional_date STRING,
    product_id STRING,
    customer_id STRING,
    payment STRING,
    credit_card STRING,
    loyalty_card STRING,
    cost STRING,
    quantity STRING,
    price STRING,
    file_name STRING,
    file_arrival_time TIMESTAMP,
    quarantine_reason STRING,
    quarantined_at TIMESTAMP
) USING DELTA;

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, when

def process_silver_microbatch(microBatchDF, batchId):
    try:
        # Rows failing basic validation: null in required fields, or non-numeric cost/price/quantity
        validated_df = microBatchDF.withColumn(
            "quarantine_reason",
            when(col("transaction_id").isNull(), "null_transaction_id")
            .when(col("price").isNull(), "null_price")
            .when(col("quantity").isNull(), "null_quantity")
            .when(col("price").cast("double").isNull() & col("price").isNotNull(), "non_numeric_price")
            .when(col("quantity").cast("int").isNull() & col("quantity").isNotNull(), "non_numeric_quantity")
            .otherwise(None)
        )

        good_df = validated_df.filter(col("quarantine_reason").isNull()).drop("quarantine_reason")
        bad_df = validated_df.filter(col("quarantine_reason").isNotNull())

        # Process good rows as before
        cleaned_df = (good_df
            .dropDuplicates(["transaction_id"])
            .withColumn("cost", col("cost").cast("double"))
            .withColumn("quantity", col("quantity").cast("int"))
            .withColumn("price", col("price").cast("double"))
            .withColumn("amount", col("price") * col("quantity"))
            .withColumn("processed_time", current_timestamp())
            .withColumn("processed_flag", lit(1)))

        cleaned_df.createOrReplaceTempView("silver_updates")
        cleaned_df.sparkSession.sql("""
            MERGE INTO delta_forge_catalog.legacy_hms_db.silver_transactions AS target
            USING silver_updates AS source
            ON target.transaction_id = source.transaction_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

        # Write bad rows to quarantine
        if bad_df.count() > 0:
            (bad_df
                .withColumn("quarantined_at", current_timestamp())
                .select("transaction_id", "transactional_date", "product_id", "customer_id",
                        "payment", "credit_card", "loyalty_card", "cost", "quantity", "price",
                        "file_name", "file_arrival_time", "quarantine_reason", "quarantined_at")
                .write.format("delta").mode("append")
                .saveAsTable("delta_forge_catalog.legacy_hms_db.silver_quarantine")
            )

        log_audit(run_id, "silver", f"batch_{batchId}", cleaned_df.count(), "SUCCESS")

    except Exception as e:
        log_audit(run_id, "silver", f"batch_{batchId}", None, "FAILED", str(e))
        raise

In [0]:
import pandas as pd

bad_data = {
    "transaction_id": ["9001", "9002", "9003"],
    "transactional_date": ["05-08-2021 10:00", "05-08-2021 10:05", "05-08-2021 10:10"],
    "product_id": ["P0100", "P0200", "P0300"],
    "customer_id": ["10", "11", "12"],
    "payment": ["visa", "mastercard", "visa"],
    "credit_card": ["4041590000000000", "5300000000000000", "4041590000000001"],
    "loyalty_card": ["F", "T", "F"],
    "cost": ["10.00", "12.50", "8.00"],
    "quantity": ["1", "N/A", "2"],       # row 2: non-numeric quantity
    "price": [None, "14.99", "9.50"]      # row 1: null price
}

df = pd.DataFrame(bad_data)

df.to_csv("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/bad_data_test.csv", index=False)

print("File written:")
print(df)

File written:
  transaction_id transactional_date product_id  ...   cost quantity  price
0           9001   05-08-2021 10:00      P0100  ...  10.00        1   None
1           9002   05-08-2021 10:05      P0200  ...  12.50      N/A  14.99
2           9003   05-08-2021 10:10      P0300  ...   8.00        2   9.50

[3 rows x 10 columns]


In [0]:
display(dbutils.fs.ls("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"))

path,name,size,modificationTime
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,Fact_Sales_1.csv,299478,1785686941000
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2.csv,Fact_Sales_2.csv,2795,1785735291000
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv,Fact_Sales_2_with_channel.csv,3321,1785735326000
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/bad_data_test.csv,bad_data_test.csv,318,1786000222000
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/late_arrival_test.csv,late_arrival_test.csv,175,1785736733000


In [0]:
from pyspark.sql.functions import col

raw_path = "/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"
checkpoint_path = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"

log_audit(run_id, "bronze", None, None, "STARTED")

try:
    df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schema_location)
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
      .option("cloudFiles.rescuedDataColumn", "_rescued_data")
      .option("header", "true")
      .load(raw_path)
      .withColumn("file_name", col("_metadata.file_path"))
      .withColumn("file_arrival_time", col("_metadata.file_modification_time"))
    )

    query = (df.writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .option("mergeSchema", "true")
      .trigger(availableNow=True)
      .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
    )
    query.awaitTermination()

    row_count = spark.sql("SELECT COUNT(*) AS c FROM delta_forge_catalog.legacy_hms_db.bronze_transactions").collect()[0]["c"]
    log_audit(run_id, "bronze", None, row_count, "SUCCESS")

except Exception as e:
    log_audit(run_id, "bronze", None, None, "FAILED", str(e))
    raise

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, when, expr

def process_silver_microbatch(microBatchDF, batchId):
    try:
        validated_df = microBatchDF.withColumn(
            "quarantine_reason",
            when(col("transaction_id").isNull(), "null_transaction_id")
            .when(col("price").isNull(), "null_price")
            .when(col("quantity").isNull(), "null_quantity")
            .when(expr("try_cast(price as double)").isNull() & col("price").isNotNull(), "non_numeric_price")
            .when(expr("try_cast(quantity as int)").isNull() & col("quantity").isNotNull(), "non_numeric_quantity")
            .otherwise(None)
        )

        good_df = validated_df.filter(col("quarantine_reason").isNull()).drop("quarantine_reason")
        bad_df = validated_df.filter(col("quarantine_reason").isNotNull())

        cleaned_df = (good_df
            .dropDuplicates(["transaction_id"])
            .withColumn("cost", expr("try_cast(cost as double)"))
            .withColumn("quantity", expr("try_cast(quantity as int)"))
            .withColumn("price", expr("try_cast(price as double)"))
            .withColumn("amount", col("price") * col("quantity"))
            .withColumn("processed_time", current_timestamp())
            .withColumn("processed_flag", lit(1)))

        cleaned_df.createOrReplaceTempView("silver_updates")
        cleaned_df.sparkSession.sql("""
            MERGE INTO delta_forge_catalog.legacy_hms_db.silver_transactions AS target
            USING silver_updates AS source
            ON target.transaction_id = source.transaction_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

        bad_count = bad_df.count()
        if bad_count > 0:
            (bad_df
                .withColumn("quarantined_at", current_timestamp())
                .select("transaction_id", "transactional_date", "product_id", "customer_id",
                        "payment", "credit_card", "loyalty_card", "cost", "quantity", "price",
                        "file_name", "file_arrival_time", "quarantine_reason", "quarantined_at")
                .write.format("delta").mode("append")
                .saveAsTable("delta_forge_catalog.legacy_hms_db.silver_quarantine")
            )

        good_count = cleaned_df.count()
        log_audit(run_id, "silver", f"batch_{batchId}", good_count, "SUCCESS")
        print(f"Batch {batchId}: {good_count} good rows, {bad_count} quarantined")

    except Exception as e:
        log_audit(run_id, "silver", f"batch_{batchId}", None, "FAILED", str(e))
        raise

In [0]:
silver_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/silver/"

bronze_stream = spark.readStream.table("delta_forge_catalog.legacy_hms_db.bronze_transactions")

(bronze_stream.writeStream
    .foreachBatch(process_silver_microbatch)
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .start()
)

In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.silver_quarantine
ORDER BY quarantined_at DESC;

transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price,file_name,file_arrival_time,quarantine_reason,quarantined_at
9001,05-08-2021 10:00,P0100,10,visa,4041590000000000,F,10.00,1,null,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/bad_data_test.csv,2026-08-06T07:10:22.000Z,null_price,2026-08-06T07:19:41.017Z
9002,05-08-2021 10:05,P0200,11,mastercard,5300000000000000,T,12.50,N/A,14.99,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/bad_data_test.csv,2026-08-06T07:10:22.000Z,non_numeric_quantity,2026-08-06T07:19:41.017Z


In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.silver_transactions
WHERE transaction_id = 9003;

transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price,amount,file_name,file_arrival_time,processed_time,processed_flag
9003,05-08-2021 10:10,P0300,12,visa,4041590000000001,F,8.0,2,9.5,19.0,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/bad_data_test.csv,2026-08-06T07:10:22.000Z,2026-08-06T07:19:37.226Z,1


In [0]:
# Check which tables survived
for t in ["bronze_transactions", "silver_transactions", "silver_quarantine", "gold_kpis", "gold_kpis_complete", "gold_batch_metrics"]:
    exists = spark.catalog.tableExists(f"delta_forge_catalog.legacy_hms_db.{t}")
    print(f"{t}: {'EXISTS (not dropped yet)' if exists else 'dropped'}")

bronze_transactions: EXISTS (not dropped yet)
silver_transactions: EXISTS (not dropped yet)
silver_quarantine: EXISTS (not dropped yet)
gold_kpis: EXISTS (not dropped yet)
gold_kpis_complete: EXISTS (not dropped yet)
gold_batch_metrics: EXISTS (not dropped yet)


In [0]:
# Check which checkpoint folders survived
for path in ["checkpoints/bronze/", "checkpoints/silver/", "checkpoints/gold/", "checkpoints/schema/"]:
    full = f"abfss://delta-forge@deltaforge.dfs.core.windows.net/{path}"
    try:
        dbutils.fs.ls(full)
        print(f"{path}: EXISTS")
    except Exception as e:
        print(f"{path}: gone or inaccessible ({str(e)[:60]})")

checkpoints/bronze/: EXISTS
checkpoints/silver/: EXISTS
checkpoints/gold/: EXISTS
checkpoints/schema/: EXISTS


In [0]:
# Check if the raw landing volume still has files
display(dbutils.fs.ls("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"))

path,name,size,modificationTime
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,Fact_Sales_1.csv,299478,1785686941000
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2.csv,Fact_Sales_2.csv,2795,1785735291000
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2_with_channel.csv,Fact_Sales_2_with_channel.csv,3321,1785735326000
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/bad_data_test.csv,bad_data_test.csv,318,1786000222000
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/late_arrival_test.csv,late_arrival_test.csv,175,1785736733000


In [0]:
# Drop all pipeline tables
spark.sql("DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.bronze_transactions")
spark.sql("DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.silver_transactions")
spark.sql("DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.silver_quarantine")
spark.sql("DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.gold_kpis")
spark.sql("DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.gold_batch_metrics")
spark.sql("DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.pipeline_audit_log")

# Clear all checkpoints
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/", recurse=True)
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/silver/", recurse=True)
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold/", recurse=True)
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/", recurse=True)

# Clear the landing volume completely (recurse=True fixes your error)
for f in dbutils.fs.ls("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"):
    dbutils.fs.rm(f.path, recurse=True)

print("Full reset done.")

Full reset done.


In [0]:
# Confirm tables are gone
for t in ["bronze_transactions", "silver_transactions", "silver_quarantine", "gold_kpis", "gold_kpis_complete", "gold_batch_metrics"]:
    exists = spark.catalog.tableExists(f"delta_forge_catalog.legacy_hms_db.{t}")
    print(f"{t}: {'STILL EXISTS ⚠️' if exists else 'dropped ✓'}")

bronze_transactions: dropped ✓
silver_transactions: dropped ✓
silver_quarantine: dropped ✓
gold_kpis: dropped ✓
gold_kpis_complete: dropped ✓
gold_batch_metrics: dropped ✓


In [0]:
# Confirm checkpoints are gone
for path in ["checkpoints/bronze/", "checkpoints/silver/", "checkpoints/gold/", "checkpoints/schema/"]:
    full = f"abfss://delta-forge@deltaforge.dfs.core.windows.net/{path}"
    try:
        dbutils.fs.ls(full)
        print(f"{path}: STILL EXISTS ⚠️")
    except Exception:
        print(f"{path}: gone ✓")

checkpoints/bronze/: gone ✓
checkpoints/silver/: gone ✓
checkpoints/gold/: gone ✓
checkpoints/schema/: gone ✓


In [0]:
# Confirm volume is empty
display(dbutils.fs.ls("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"))

[]

In [0]:
%sql
CREATE TABLE delta_forge_catalog.legacy_hms_db.silver_transactions (
    transaction_id STRING,
    transactional_date STRING,
    product_id STRING,
    customer_id STRING,
    payment STRING,
    credit_card STRING,
    loyalty_card STRING,
    cost DOUBLE,
    quantity INT,
    price DOUBLE,
    amount DOUBLE,
    file_name STRING,
    file_arrival_time TIMESTAMP,
    processed_time TIMESTAMP,
    processed_flag INT
) USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delta_forge_catalog.legacy_hms_db.silver_quarantine (
    transaction_id STRING,
    transactional_date STRING,
    product_id STRING,
    customer_id STRING,
    payment STRING,
    credit_card STRING,
    loyalty_card STRING,
    cost STRING,
    quantity STRING,
    price STRING,
    file_name STRING,
    file_arrival_time TIMESTAMP,
    quarantine_reason STRING,
    quarantined_at TIMESTAMP
) USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delta_forge_catalog.legacy_hms_db.gold_kpis (
    window_start TIMESTAMP,
    window_end TIMESTAMP,
    total_records_processed BIGINT,
    total_transaction_value DOUBLE,
    new_files_ingested BIGINT,
    duplicate_records_eliminated BIGINT,
    latest_data_freshness TIMESTAMP,
    freshness_lag_minutes DOUBLE,
    kpi_computed_at TIMESTAMP
) USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delta_forge_catalog.legacy_hms_db.gold_batch_metrics (
    batch_id BIGINT,
    input_rows BIGINT,
    duration_ms BIGINT,
    recorded_at TIMESTAMP
) USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delta_forge_catalog.legacy_hms_db.pipeline_audit_log (
    run_id          STRING,
    layer           STRING,
    file_name       STRING,
    row_count       LONG,
    status          STRING,
    error_message   STRING,
    run_timestamp   TIMESTAMP
) USING DELTA;

In [0]:
print(spark.catalog.tableExists("delta_forge_catalog.legacy_hms_db.pipeline_audit_log"))

True


In [0]:
for t in ["silver_transactions", "silver_quarantine", "gold_kpis", "gold_batch_metrics"]:
    print(t, spark.catalog.tableExists(f"delta_forge_catalog.legacy_hms_db.{t}"))

silver_transactions True
silver_quarantine True
gold_kpis True
gold_batch_metrics True


In [0]:
display(dbutils.fs.ls("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/"))

path,name,size,modificationTime
dbfs:/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,Fact_Sales_1.csv,299478,1786283630000


In [0]:
display(dbutils.fs.ls("abfss://delta-forge@deltaforge.dfs.core.windows.net/raw_landing/"))

path,name,size,modificationTime
abfss://delta-forge@deltaforge.dfs.core.windows.net/raw_landing/Fact_Sales_1.csv,Fact_Sales_1.csv,299478,1786105613000


In [0]:
from pyspark.sql.functions import current_timestamp, lit, input_file_name,col
import uuid


run_id = str(uuid.uuid4())

def log_audit(layer, file_name, row_count, status, error_message=None):
    spark.sql(f"""
        INSERT INTO delta_forge_catalog.legacy_hms_db.pipeline_audit_log
        VALUES ('{run_id}', '{layer}', {f"'{file_name}'" if file_name else 'NULL'},
                {row_count if row_count is not None else 'NULL'}, '{status}',
                {f"'{error_message}'" if error_message else 'NULL'}, current_timestamp())
    """)

bronze_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"

log_audit("bronze", None, None, "STARTED")

try:
    bronze_df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .option("rescuedDataColumn", "_rescued_data")
        .load("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/")
        .withColumn("file_name", col("_metadata.file_path"))
        .withColumn("ingested_at", current_timestamp())
    )

    (bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", bronze_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
        .awaitTermination()
    )

    row_count = spark.table("delta_forge_catalog.legacy_hms_db.bronze_transactions").count()
    log_audit("bronze", None, row_count, "SUCCESS")
    print(f"Bronze SUCCESS — total rows: {row_count}")

except Exception as e:
    log_audit("bronze", None, None, "FAILED", str(e)[:500])
    raise

Bronze SUCCESS — total rows: 4410


In [0]:
%sql
DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.silver_transactions;

CREATE TABLE delta_forge_catalog.legacy_hms_db.silver_transactions (
    transaction_id       STRING,
    transactional_date    STRING,
    product_id            STRING,
    customer_id           STRING,
    payment               STRING,
    credit_card           STRING,
    loyalty_card          STRING,
    cost                   DOUBLE,
    quantity               DOUBLE,
    price                  DOUBLE,
    file_name              STRING,
    ingested_at            TIMESTAMP,
    processed_time         TIMESTAMP
) USING DELTA;

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import col, lit, current_timestamp, row_number
from pyspark.sql.window import Window

audit_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("row_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
])

def log_audit(layer, file_name, row_count, status, error_message=None):
    row = Row(
        run_id=run_id,
        layer=layer,
        file_name=file_name,
        row_count=int(row_count) if row_count is not None else None,
        status=status,
        error_message=error_message
    )
    df = spark.createDataFrame([row], schema=audit_schema).withColumn("run_timestamp", current_timestamp())
    df.write.format("delta").mode("append").saveAsTable(
        "delta_forge_catalog.legacy_hms_db.pipeline_audit_log"
    )

log_audit("silver", None, None, "STARTED")

try:
    bronze_batch = spark.table("delta_forge_catalog.legacy_hms_db.bronze_transactions")

    valid_df = bronze_batch.filter(col("price").cast("double").isNotNull())
    quarantine_df = (bronze_batch.filter(col("price").cast("double").isNull())
        .withColumn("quarantine_reason", lit("non_numeric_price"))
        .withColumn("quarantined_at", current_timestamp())
    )

    w = Window.partitionBy("transaction_id").orderBy(col("ingested_at").desc())
    deduped_df = (valid_df
        .withColumn("rn", row_number().over(w))
        .filter(col("rn") == 1)
        .drop("rn")
        .withColumn("processed_time", current_timestamp())
    )

    deduped_df.createOrReplaceTempView("silver_updates")
    spark.sql("""
        MERGE INTO delta_forge_catalog.legacy_hms_db.silver_transactions AS target
        USING silver_updates AS source
        ON target.transaction_id = source.transaction_id
        WHEN MATCHED THEN UPDATE SET
            transaction_id = source.transaction_id,
            transactional_date = source.transactional_date,
            product_id = source.product_id,
            customer_id = source.customer_id,
            payment = source.payment,
            credit_card = source.credit_card,
            loyalty_card = source.loyalty_card,
            cost = source.cost,
            quantity = source.quantity,
            price = source.price,
            file_name = source.file_name,
            ingested_at = source.ingested_at,
            processed_time = source.processed_time
        WHEN NOT MATCHED THEN INSERT (
            transaction_id, transactional_date, product_id, customer_id, payment,
            credit_card, loyalty_card, cost, quantity, price, file_name, ingested_at, processed_time
        ) VALUES (
            source.transaction_id, source.transactional_date, source.product_id, source.customer_id,
            source.payment, source.credit_card, source.loyalty_card, source.cost, source.quantity,
            source.price, source.file_name, source.ingested_at, source.processed_time
        )
    """)

    quarantine_df.select(
        "transaction_id", "cost", "price", "quantity", "file_name",
        "quarantine_reason", "quarantined_at"
    ).write.format("delta").mode("append").saveAsTable(
        "delta_forge_catalog.legacy_hms_db.silver_quarantine"
    )

    row_count = spark.table("delta_forge_catalog.legacy_hms_db.silver_transactions").count()
    log_audit("silver", None, row_count, "SUCCESS")
    print(f"Silver SUCCESS — total rows: {row_count}")

except Exception as e:
    log_audit("silver", None, None, "FAILED", str(e)[:2000])
    raise

Silver SUCCESS — total rows: 4410


In [0]:
print(deduped_df.count())

4410


In [0]:
%sql DESCRIBE HISTORY delta_forge_catalog.legacy_hms_db.silver_transactions

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-08-10T11:35:55.000Z,142299825447056,1000021511@dit.edu.in,MERGE,"Map(predicate -> [""(transaction_id#14160 = transaction_id#14126)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(303522912617808),c5c47ab2-642c-42d2-a9e2-acc80cbfd663,0810-112526-qb72auk1-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 54813, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2029, materializeSourceTimeMs -> 406, numTargetRowsInserted -> 4410, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 563, numTargetRowsUpdated -> 0, numOutputRows -> 4410, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4410, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1033)",null,Databricks-Runtime/18.x-photon-scala2.13
0,2026-08-10T11:35:38.000Z,142299825447056,1000021511@dit.edu.in,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-102faeca-86ca-43a0-95da-b0ccf15fe4bb"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-19a61ca1-cee7-404b-b431-a8c092af338e""}, statsOnLoad -> false)",null,List(303522912617808),8e819c43-ef96-46ee-895b-f0537f2386c9,0810-112526-qb72auk1-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-photon-scala2.13


In [0]:
%sql
SELECT 'bronze' AS layer, COUNT(*) AS row_count FROM delta_forge_catalog.legacy_hms_db.bronze_transactions
UNION ALL
SELECT 'silver', COUNT(*) FROM delta_forge_catalog.legacy_hms_db.silver_transactions


layer,row_count
bronze,4410
silver,4410


In [0]:
# Drop the table completely
spark.sql("DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.gold_kpis")

# Remove old checkpoint(s) — both possible paths
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold/", recurse=True)
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold_v2/", recurse=True)

False

In [0]:
%sql
CREATE TABLE delta_forge_catalog.legacy_hms_db.gold_kpis (
    window_start                 TIMESTAMP,
    window_end                   TIMESTAMP,
    total_records_processed      LONG,
    total_transaction_value      DOUBLE,
    new_files_ingested           LONG,
    duplicate_records_eliminated LONG,
    latest_data_freshness        TIMESTAMP,
    kpi_computed_at              TIMESTAMP,
    freshness_lag_minutes        DOUBLE
) USING DELTA;

In [0]:
from pyspark.sql.functions import window, count, sum as _sum, max as _max, approx_count_distinct, unix_timestamp, col, lit, current_timestamp

# --- Reset table + checkpoint fully before running ---
spark.sql("DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.gold_kpis")
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold_fresh/", recurse=True)

spark.sql("""
CREATE TABLE delta_forge_catalog.legacy_hms_db.gold_kpis (
    window_start                 TIMESTAMP,
    window_end                   TIMESTAMP,
    total_records_processed      LONG,
    total_transaction_value      DOUBLE,
    new_files_ingested           LONG,
    duplicate_records_eliminated LONG,
    latest_data_freshness        TIMESTAMP,
    kpi_computed_at              TIMESTAMP,
    freshness_lag_minutes        DOUBLE
) USING DELTA
""")

# --- Build the streaming aggregation ---
gold_df = (spark.readStream.table("delta_forge_catalog.legacy_hms_db.silver_transactions")
    .withWatermark("processed_time", "10 minutes")
    .groupBy(window("processed_time", "1 hour"))
    .agg(
        count("transaction_id").alias("total_records_processed"),
        _sum("cost").alias("total_transaction_value"),
        approx_count_distinct("file_name").alias("new_files_ingested"),
        _max("processed_time").alias("latest_data_freshness")
    )
    .withColumn("window_start", col("window.start"))
    .withColumn("window_end", col("window.end"))
    .withColumn("kpi_computed_at", current_timestamp())
    .withColumn("freshness_lag_minutes", (unix_timestamp(current_timestamp()) - unix_timestamp(col("latest_data_freshness"))) / 60)
    .withColumn("duplicate_records_eliminated", lit(0))
    .drop("window")
)

# --- Upsert each micro-batch into gold_kpis via MERGE ---
def upsert_gold_kpis(batch_df, batch_id):
    if batch_df.count() == 0:
        print(f"Batch {batch_id}: 0 rows, skipping merge")
        return
    batch_df.createOrReplaceTempView("gold_updates")
    batch_df.sparkSession.sql("""
        MERGE INTO delta_forge_catalog.legacy_hms_db.gold_kpis AS target
        USING gold_updates AS source
        ON target.window_start = source.window_start AND target.window_end = source.window_end
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Batch {batch_id}: merged {batch_df.count()} rows")

gold_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold_fresh/"

(gold_df.writeStream
    .outputMode("update")
    .foreachBatch(upsert_gold_kpis)
    .option("checkpointLocation", gold_checkpoint)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

# --- Verify immediately ---
row_count = spark.table("delta_forge_catalog.legacy_hms_db.gold_kpis").count()
print(f"Gold SUCCESS — total rows: {row_count}")

Gold SUCCESS — total rows: 1


In [0]:
%sql
SELECT 'bronze' AS layer, COUNT(*) AS row_count FROM delta_forge_catalog.legacy_hms_db.bronze_transactions
UNION ALL
SELECT 'silver', COUNT(*) FROM delta_forge_catalog.legacy_hms_db.silver_transactions
UNION ALL
SELECT 'gold_kpis', SUM(total_records_processed) FROM delta_forge_catalog.legacy_hms_db.gold_kpis;

layer,row_count
gold_kpis,4410
bronze,4410
silver,4410


In [0]:
from pyspark.sql.functions import current_timestamp, lit, input_file_name,col
import uuid


run_id = str(uuid.uuid4())

def log_audit(layer, file_name, row_count, status, error_message=None):
    spark.sql(f"""
        INSERT INTO delta_forge_catalog.legacy_hms_db.pipeline_audit_log
        VALUES ('{run_id}', '{layer}', {f"'{file_name}'" if file_name else 'NULL'},
                {row_count if row_count is not None else 'NULL'}, '{status}',
                {f"'{error_message}'" if error_message else 'NULL'}, current_timestamp())
    """)

bronze_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"

log_audit("bronze", None, None, "STARTED")

try:
    bronze_df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .option("rescuedDataColumn", "_rescued_data")
        .load("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/")
        .withColumn("file_name", col("_metadata.file_path"))
        .withColumn("ingested_at", current_timestamp())
    )

    (bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", bronze_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
        .awaitTermination()
    )

    row_count = spark.table("delta_forge_catalog.legacy_hms_db.bronze_transactions").count()
    log_audit("bronze", None, row_count, "SUCCESS")
    print(f"Bronze SUCCESS — total rows: {row_count}")

except Exception as e:
    log_audit("bronze", None, None, "FAILED", str(e)[:500])
    raise

Bronze SUCCESS — total rows: 7100


In [0]:
%sql
DESCRIBE delta_forge_catalog.legacy_hms_db.silver_transactions;

col_name,data_type,comment
transaction_id,string,null
transactional_date,string,null
product_id,string,null
customer_id,string,null
payment,string,null
credit_card,string,null
loyalty_card,string,null
cost,double,null
quantity,int,null
price,double,null


In [0]:
from pyspark.sql.functions import window, count, sum as _sum, max as _max, approx_count_distinct, unix_timestamp, col, lit, current_timestamp

# --- Reset table + checkpoint fully before running ---
spark.sql("DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.gold_kpis")
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold_fresh/", recurse=True)

spark.sql("""
CREATE TABLE delta_forge_catalog.legacy_hms_db.gold_kpis (
    window_start                 TIMESTAMP,
    window_end                   TIMESTAMP,
    total_records_processed      LONG,
    total_transaction_value      DOUBLE,
    new_files_ingested           LONG,
    duplicate_records_eliminated LONG,
    latest_data_freshness        TIMESTAMP,
    kpi_computed_at              TIMESTAMP,
    freshness_lag_minutes        DOUBLE
) USING DELTA
""")

# --- Build the streaming aggregation ---
gold_df = (spark.readStream.table("delta_forge_catalog.legacy_hms_db.silver_transactions")
    .withWatermark("processed_time", "10 minutes")
    .groupBy(window("processed_time", "1 hour"))
    .agg(
        count("transaction_id").alias("total_records_processed"),
        _sum("cost").alias("total_transaction_value"),
        approx_count_distinct("file_name").alias("new_files_ingested"),
        _max("processed_time").alias("latest_data_freshness")
    )
    .withColumn("window_start", col("window.start"))
    .withColumn("window_end", col("window.end"))
    .withColumn("kpi_computed_at", current_timestamp())
    .withColumn("freshness_lag_minutes", (unix_timestamp(current_timestamp()) - unix_timestamp(col("latest_data_freshness"))) / 60)
    .withColumn("duplicate_records_eliminated", lit(0))
    .drop("window")
)

# --- Upsert each micro-batch into gold_kpis via MERGE ---
def upsert_gold_kpis(batch_df, batch_id):
    if batch_df.count() == 0:
        print(f"Batch {batch_id}: 0 rows, skipping merge")
        return
    batch_df.createOrReplaceTempView("gold_updates")
    batch_df.sparkSession.sql("""
        MERGE INTO delta_forge_catalog.legacy_hms_db.gold_kpis AS target
        USING gold_updates AS source
        ON target.window_start = source.window_start AND target.window_end = source.window_end
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Batch {batch_id}: merged {batch_df.count()} rows")

gold_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold_fresh/"

(gold_df.writeStream
    .outputMode("update")
    .foreachBatch(upsert_gold_kpis)
    .option("checkpointLocation", gold_checkpoint)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

# --- Verify immediately ---
row_count = spark.table("delta_forge_catalog.legacy_hms_db.gold_kpis").count()
print(f"Gold SUCCESS — total rows: {row_count}")

Gold SUCCESS — total rows: 0


In [0]:
%sql
SELECT file_name, COUNT(*) AS row_count
FROM delta_forge_catalog.legacy_hms_db.bronze_transactions
GROUP BY file_name;

file_name,row_count
/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_2.csv,40
/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv,4410


In [0]:
from pyspark.sql.functions import current_timestamp, lit, input_file_name,col
import uuid


run_id = str(uuid.uuid4())

def log_audit(layer, file_name, row_count, status, error_message=None):
    spark.sql(f"""
        INSERT INTO delta_forge_catalog.legacy_hms_db.pipeline_audit_log
        VALUES ('{run_id}', '{layer}', {f"'{file_name}'" if file_name else 'NULL'},
                {row_count if row_count is not None else 'NULL'}, '{status}',
                {f"'{error_message}'" if error_message else 'NULL'}, current_timestamp())
    """)

bronze_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"

log_audit("bronze", None, None, "STARTED")

try:
    bronze_df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .option("rescuedDataColumn", "_rescued_data")
        .load("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/")
        .withColumn("file_name", col("_metadata.file_path"))
        .withColumn("ingested_at", current_timestamp())
    )

    (bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", bronze_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
        .awaitTermination()
    )

    row_count = spark.table("delta_forge_catalog.legacy_hms_db.bronze_transactions").count()
    log_audit("bronze", None, row_count, "SUCCESS")
    print(f"Bronze SUCCESS — total rows: {row_count}")

except Exception as e:
    log_audit("bronze", None, None, "FAILED", str(e)[:500])
    raise

Bronze SUCCESS — total rows: 4450


In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.pipeline_audit_log
WHERE layer = 'bronze'
ORDER BY run_timestamp DESC
LIMIT 5;

run_id,layer,file_name,row_count,status,error_message,run_timestamp
a5a48853-271b-4cfa-8173-e77bacba890d,bronze,null,4450,SUCCESS,null,2026-08-10T12:16:52.198Z
a5a48853-271b-4cfa-8173-e77bacba890d,bronze,null,null,STARTED,null,2026-08-10T12:16:48.852Z
8b4747f3-07c6-48c3-9240-0f13c2bb6ff3,bronze,null,4450,SUCCESS,null,2026-08-10T12:16:44.785Z
8b4747f3-07c6-48c3-9240-0f13c2bb6ff3,bronze,null,null,STARTED,null,2026-08-10T12:16:40.784Z
bd4fb464-9975-457f-93d6-d6ba9f45eb12,bronze,null,4450,SUCCESS,null,2026-08-10T12:13:16.088Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import col, lit, current_timestamp, row_number
from pyspark.sql.window import Window

audit_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("row_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
])

def log_audit(layer, file_name, row_count, status, error_message=None):
    row = Row(
        run_id=run_id,
        layer=layer,
        file_name=file_name,
        row_count=int(row_count) if row_count is not None else None,
        status=status,
        error_message=error_message
    )
    df = spark.createDataFrame([row], schema=audit_schema).withColumn("run_timestamp", current_timestamp())
    df.write.format("delta").mode("append").saveAsTable(
        "delta_forge_catalog.legacy_hms_db.pipeline_audit_log"
    )

log_audit("silver", None, None, "STARTED")

try:
    bronze_batch = spark.table("delta_forge_catalog.legacy_hms_db.bronze_transactions")

    valid_df = bronze_batch.filter(col("price").cast("double").isNotNull())
    quarantine_df = (bronze_batch.filter(col("price").cast("double").isNull())
        .withColumn("quarantine_reason", lit("non_numeric_price"))
        .withColumn("quarantined_at", current_timestamp())
    )

    w = Window.partitionBy("transaction_id").orderBy(col("ingested_at").desc())
    deduped_df = (valid_df
        .withColumn("rn", row_number().over(w))
        .filter(col("rn") == 1)
        .drop("rn")
        .withColumn("processed_time", current_timestamp())
    )

    deduped_df.createOrReplaceTempView("silver_updates")
    spark.sql("""
        MERGE INTO delta_forge_catalog.legacy_hms_db.silver_transactions AS target
        USING silver_updates AS source
        ON target.transaction_id = source.transaction_id
        WHEN MATCHED THEN UPDATE SET
            transaction_id = source.transaction_id,
            transactional_date = source.transactional_date,
            product_id = source.product_id,
            customer_id = source.customer_id,
            payment = source.payment,
            credit_card = source.credit_card,
            loyalty_card = source.loyalty_card,
            cost = source.cost,
            quantity = source.quantity,
            price = source.price,
            file_name = source.file_name,
            ingested_at = source.ingested_at,
            processed_time = source.processed_time
        WHEN NOT MATCHED THEN INSERT (
            transaction_id, transactional_date, product_id, customer_id, payment,
            credit_card, loyalty_card, cost, quantity, price, file_name, ingested_at, processed_time
        ) VALUES (
            source.transaction_id, source.transactional_date, source.product_id, source.customer_id,
            source.payment, source.credit_card, source.loyalty_card, source.cost, source.quantity,
            source.price, source.file_name, source.ingested_at, source.processed_time
        )
    """)

    quarantine_df.select(
        "transaction_id", "cost", "price", "quantity", "file_name",
        "quarantine_reason", "quarantined_at"
    ).write.format("delta").mode("append").saveAsTable(
        "delta_forge_catalog.legacy_hms_db.silver_quarantine"
    )

    row_count = spark.table("delta_forge_catalog.legacy_hms_db.silver_transactions").count()
    log_audit("silver", None, row_count, "SUCCESS")
    print(f"Silver SUCCESS — total rows: {row_count}")

except Exception as e:
    log_audit("silver", None, None, "FAILED", str(e)[:2000])
    raise

Silver SUCCESS — total rows: 4450


In [0]:
from pyspark.sql.functions import window, count, sum as _sum, max as _max, approx_count_distinct, unix_timestamp, col, lit, current_timestamp

# --- Reset table + checkpoint fully before running ---
spark.sql("DROP TABLE IF EXISTS delta_forge_catalog.legacy_hms_db.gold_kpis")
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold_fresh/", recurse=True)

spark.sql("""
CREATE TABLE delta_forge_catalog.legacy_hms_db.gold_kpis (
    window_start                 TIMESTAMP,
    window_end                   TIMESTAMP,
    total_records_processed      LONG,
    total_transaction_value      DOUBLE,
    new_files_ingested           LONG,
    duplicate_records_eliminated LONG,
    latest_data_freshness        TIMESTAMP,
    kpi_computed_at              TIMESTAMP,
    freshness_lag_minutes        DOUBLE
) USING DELTA
""")

# --- Build the streaming aggregation ---
gold_df = (spark.readStream.table("delta_forge_catalog.legacy_hms_db.silver_transactions")
    .withWatermark("processed_time", "10 minutes")
    .groupBy(window("processed_time", "1 hour"))
    .agg(
        count("transaction_id").alias("total_records_processed"),
        _sum("cost").alias("total_transaction_value"),
        approx_count_distinct("file_name").alias("new_files_ingested"),
        _max("processed_time").alias("latest_data_freshness")
    )
    .withColumn("window_start", col("window.start"))
    .withColumn("window_end", col("window.end"))
    .withColumn("kpi_computed_at", current_timestamp())
    .withColumn("freshness_lag_minutes", (unix_timestamp(current_timestamp()) - unix_timestamp(col("latest_data_freshness"))) / 60)
    .withColumn("duplicate_records_eliminated", lit(0))
    .drop("window")
)

# --- Upsert each micro-batch into gold_kpis via MERGE ---
def upsert_gold_kpis(batch_df, batch_id):
    if batch_df.count() == 0:
        print(f"Batch {batch_id}: 0 rows, skipping merge")
        return
    batch_df.createOrReplaceTempView("gold_updates")
    batch_df.sparkSession.sql("""
        MERGE INTO delta_forge_catalog.legacy_hms_db.gold_kpis AS target
        USING gold_updates AS source
        ON target.window_start = source.window_start AND target.window_end = source.window_end
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Batch {batch_id}: merged {batch_df.count()} rows")

gold_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/gold_fresh/"

(gold_df.writeStream
    .outputMode("update")
    .foreachBatch(upsert_gold_kpis)
    .option("checkpointLocation", gold_checkpoint)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

# --- Verify immediately ---
row_count = spark.table("delta_forge_catalog.legacy_hms_db.gold_kpis").count()
print(f"Gold SUCCESS — total rows: {row_count}")

Gold SUCCESS — total rows: 1


In [0]:
%sql
SELECT 'bronze' AS layer, COUNT(*) AS row_count FROM delta_forge_catalog.legacy_hms_db.bronze_transactions
UNION ALL
SELECT 'silver', COUNT(*) FROM delta_forge_catalog.legacy_hms_db.silver_transactions
UNION ALL
SELECT 'gold_kpis', SUM(total_records_processed) FROM delta_forge_catalog.legacy_hms_db.gold_kpis;

layer,row_count
gold_kpis,4450
bronze,4450
silver,4450


In [0]:
import pandas as pd

preview = pd.read_csv("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv", nrows=5)
print("Columns:", preview.columns.tolist())
print()
print(preview)

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

   InvoiceNo  StockCode  ... CustomerID         Country
0     537667      22158  ...      17870  United Kingdom
1     537668      22867  ...      14821  United Kingdom
2     537668      22158  ...      14821  United Kingdom
3     537669      84978  ...      16863  United Kingdom
4     537669      21726  ...      16863  United Kingdom

[5 rows x 8 columns]


In [0]:
import pandas as pd

df = pd.read_csv("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/Fact_Sales_1.csv")
truncated = df.head(3).copy()
truncated.loc[0, "price"] = "###CORRUPTED###"   # forces a real parse/validation issue

truncated.to_csv("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/truncated_test.csv", index=False)
print("Corrupted test file written.")
print(truncated)

Corrupted test file written.
   transaction_id transactional_date  ... quantity            price
0               1   04-05-2021 02:00  ...        2  ###CORRUPTED###
1               2   04-05-2021 03:04  ...        1             1.49
2               3   04-05-2021 03:56  ...        3             5.89

[3 rows x 10 columns]


/home/spark-c5e1eb31-0beb-4fbe-96ab-5a/.ipykernel/88/command-6323355832576533-1543514655:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '###CORRUPTED###' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  truncated.loc[0, "price"] = "###CORRUPTED###"   # forces a real parse/validation issue


In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.bronze_transactions WHERE _rescued_data IS NOT NULL;

transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price,_rescued_data,file_name,ingested_at,InvoiceNo,StockCode,Description,InvoiceDate,UnitPrice,CustomerID,Country
null,null,null,null,null,null,null,null,null,null,"{""Quantity"":""128"",""_file_path"":""/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv""}",/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv,2026-08-10T12:30:41.331Z,537667,22158,3 HEARTS HANGING DECORATION RUSTIC,08-12-2010 08:12,2.55,17870,United Kingdom
null,null,null,null,null,null,null,null,null,null,"{""Quantity"":""12"",""_file_path"":""/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv""}",/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv,2026-08-10T12:30:41.331Z,537668,22867,HAND WARMER BIRD DESIGN,08-12-2010 08:43,2.1,14821,United Kingdom
null,null,null,null,null,null,null,null,null,null,"{""Quantity"":""8"",""_file_path"":""/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv""}",/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv,2026-08-10T12:30:41.331Z,537668,22158,3 HEARTS HANGING DECORATION RUSTIC,08-12-2010 08:43,2.95,14821,United Kingdom
null,null,null,null,null,null,null,null,null,null,"{""Quantity"":""12"",""_file_path"":""/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv""}",/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv,2026-08-10T12:30:41.331Z,537669,84978,HANGING HEART JAR T-LIGHT HOLDER,08-12-2010 08:58,1.25,16863,United Kingdom
null,null,null,null,null,null,null,null,null,null,"{""Quantity"":""12"",""_file_path"":""/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv""}",/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv,2026-08-10T12:30:41.331Z,537669,21726,MULTI HEARTS STICKERS,08-12-2010 08:58,0.85,16863,United Kingdom
null,null,null,null,null,null,null,null,null,null,"{""Quantity"":""12"",""_file_path"":""/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv""}",/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv,2026-08-10T12:30:41.331Z,537669,21723,ALPHABET HEARTS STICKER SHEET,08-12-2010 08:58,0.85,16863,United Kingdom
null,null,null,null,null,null,null,null,null,null,"{""Quantity"":""12"",""_file_path"":""/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv""}",/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv,2026-08-10T12:30:41.331Z,537669,21725,SWEETIES STICKERS,08-12-2010 08:58,0.85,16863,United Kingdom
null,null,null,null,null,null,null,null,null,null,"{""Quantity"":""12"",""_file_path"":""/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv""}",/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv,2026-08-10T12:30:41.331Z,537669,22969,HOMEMADE JAM SCENTED CANDLES,08-12-2010 08:58,1.45,16863,United Kingdom
null,null,null,null,null,null,null,null,null,null,"{""Quantity"":""9"",""_file_path"":""/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv""}",/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv,2026-08-10T12:30:41.331Z,537669,72741,GRAND CHOCOLATECANDLE,08-12-2010 08:58,1.45,16863,United Kingdom
null,null,null,null,null,null,null,null,null,null,"{""Quantity"":""12"",""_file_path"":""/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv""}",/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv,2026-08-10T12:30:41.331Z,537669,22644,CERAMIC CHERRY CAKE MONEY BANK,08-12-2010 08:58,1.45,16863,United Kingdom


In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.pipeline_audit_log
ORDER BY run_timestamp DESC
LIMIT 5;

run_id,layer,file_name,row_count,status,error_message,run_timestamp
be0bc779-7c9f-44e1-a307-6834f9b06233,bronze,null,null,FAILED,"[STREAM_FAILED] Query [id = cd43415a-9a58-4733-84a3-80050e5b6ede, runId = 2bb697fe-36b4-4693-b891-b6da8d8c6a1d] terminated with exception: [UNKNOWN_FIELD_EXCEPTION.NEW_FIELDS_IN_FILE] Encountered unknown fields during parsing: [InvoiceNo, StockCode, Description, InvoiceDate, UnitPrice, CustomerID, Country], which can be fixed by an automatic retry: true Unknown fields inside file path: /Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv Rescued file schema: InvoiceNo STRING,Stock",2026-08-10T12:30:45.710Z
be0bc779-7c9f-44e1-a307-6834f9b06233,bronze,null,null,STARTED,null,2026-08-10T12:30:27.305Z
a5a48853-271b-4cfa-8173-e77bacba890d,silver,null,4450,SUCCESS,null,2026-08-10T12:18:46.397Z
a5a48853-271b-4cfa-8173-e77bacba890d,silver,null,null,STARTED,null,2026-08-10T12:18:39.884Z
a5a48853-271b-4cfa-8173-e77bacba890d,bronze,null,4450,SUCCESS,null,2026-08-10T12:16:52.198Z


In [0]:
%sql
SELECT COUNT(*) FROM delta_forge_catalog.legacy_hms_db.bronze_transactions 
WHERE file_name LIKE '%truncated_test%';

COUNT(*)
3


In [0]:
%sql
SELECT price, TRY_CAST(price AS DOUBLE) AS cast_result
FROM delta_forge_catalog.legacy_hms_db.bronze_transactions
WHERE file_name LIKE '%truncated_test%';

price,cast_result
###CORRUPTED###,null
1.49,1.49
5.89,5.89


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import col, lit, expr, current_timestamp, row_number
from pyspark.sql.window import Window

audit_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("row_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
])

def log_audit(layer, file_name, row_count, status, error_message=None):
    row = Row(
        run_id=run_id,
        layer=layer,
        file_name=file_name,
        row_count=int(row_count) if row_count is not None else None,
        status=status,
        error_message=error_message
    )
    df = spark.createDataFrame([row], schema=audit_schema).withColumn("run_timestamp", current_timestamp())
    df.write.format("delta").mode("append").saveAsTable(
        "delta_forge_catalog.legacy_hms_db.pipeline_audit_log"
    )

log_audit("silver", None, None, "STARTED")

try:
    bronze_batch = spark.table("delta_forge_catalog.legacy_hms_db.bronze_transactions")

    # Exclude rows from the incompatible UCI dataset (InvoiceNo present) if any slipped in
    bronze_batch = bronze_batch.filter(col("InvoiceNo").isNull())

    valid_df = bronze_batch.filter(expr("try_cast(price AS DOUBLE)").isNotNull())
    quarantine_df = (bronze_batch.filter(expr("try_cast(price AS DOUBLE)").isNull())
        .withColumn("quarantine_reason", lit("non_numeric_price"))
        .withColumn("quarantined_at", current_timestamp())
    )

    w = Window.partitionBy("transaction_id").orderBy(col("ingested_at").desc())
    deduped_df = (valid_df
        .withColumn("rn", row_number().over(w))
        .filter(col("rn") == 1)
        .drop("rn")
        .withColumn("cost", expr("try_cast(cost AS DOUBLE)"))
        .withColumn("quantity", expr("try_cast(quantity AS INT)"))
        .withColumn("price", expr("try_cast(price AS DOUBLE)"))
        .withColumn("amount", expr("try_cast(cost AS DOUBLE) * try_cast(quantity AS DOUBLE)"))
        .withColumn("processed_time", current_timestamp())
        .withColumn("processed_flag", lit(1))
        .withColumnRenamed("ingested_at", "file_arrival_time")
    )

    deduped_df.createOrReplaceTempView("silver_updates")
    spark.sql("""
        MERGE INTO delta_forge_catalog.legacy_hms_db.silver_transactions AS target
        USING silver_updates AS source
        ON target.transaction_id = source.transaction_id
        WHEN MATCHED THEN UPDATE SET
            transaction_id = source.transaction_id,
            transactional_date = source.transactional_date,
            product_id = source.product_id,
            customer_id = source.customer_id,
            payment = source.payment,
            credit_card = source.credit_card,
            loyalty_card = source.loyalty_card,
            cost = source.cost,
            quantity = source.quantity,
            price = source.price,
            amount = source.amount,
            file_name = source.file_name,
            file_arrival_time = source.file_arrival_time,
            processed_time = source.processed_time,
            processed_flag = source.processed_flag
        WHEN NOT MATCHED THEN INSERT (
            transaction_id, transactional_date, product_id, customer_id, payment,
            credit_card, loyalty_card, cost, quantity, price, amount, file_name,
            file_arrival_time, processed_time, processed_flag
        ) VALUES (
            source.transaction_id, source.transactional_date, source.product_id, source.customer_id,
            source.payment, source.credit_card, source.loyalty_card, source.cost, source.quantity,
            source.price, source.amount, source.file_name, source.file_arrival_time,
            source.processed_time, source.processed_flag
        )
    """)

    quarantine_df.select(
        "transaction_id", "cost", "price", "quantity", "file_name",
        "quarantine_reason", "quarantined_at"
    ).write.format("delta").mode("append").saveAsTable(
        "delta_forge_catalog.legacy_hms_db.silver_quarantine"
    )

    row_count = spark.table("delta_forge_catalog.legacy_hms_db.silver_transactions").count()
    log_audit("silver", None, row_count, "SUCCESS")
    print(f"Silver SUCCESS — total rows: {row_count}")

except Exception as e:
    log_audit("silver", None, None, "FAILED", str(e)[:2000])
    raise

Silver SUCCESS — total rows: 4450


In [0]:
%sql
SELECT transaction_id, price, quarantine_reason, file_name
FROM delta_forge_catalog.legacy_hms_db.silver_quarantine
WHERE file_name LIKE '%truncated_test%';

transaction_id,price,quarantine_reason,file_name
1,###CORRUPTED###,non_numeric_price,/Volumes/delta-forge-catalog/bronze/raw_landing_vol/truncated_test.csv


In [0]:
%sql
SELECT * FROM delta_forge_catalog.legacy_hms_db.pipeline_audit_log
   WHERE status = 'FAILED'
   ORDER BY run_timestamp DESC
   LIMIT 5;

run_id,layer,file_name,row_count,status,error_message,run_timestamp
4fc616ca-cf1d-47e4-ba0e-1aebe0171bb0,silver,null,null,FAILED,"[CAST_INVALID_INPUT] The value '###CORRUPTED###' of the type ""STRING"" cannot be cast to ""DOUBLE"" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018 JVM stacktrace: org.apache.spark.SparkNumberFormatException at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:189) at com.databricks.photon.PhotonException$.getSparkException(PhotonException.scala:271) at com.databricks.photon.PhotonWriteResultHandler.getResult(PhotonWriteStageExec.scala:153) at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.open(PhotonBasicEvaluatorFactory.scala:264) at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.hasNextImpl(PhotonBasicEvaluatorFactory.scala:269) at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.$anonfun$hasNext$1(PhotonBasicEvaluatorFactory.scala:289) at scala.runtime.java8.JFunction0$mcZ$sp.apply(JFunction0$mcZ$sp.scala:17) at com.databricks.photon.metrics.BillableTimeTaskMetrics.withPhotonBilling(BillableTimeTaskMetrics.scala:71) at org.apache.spark.TaskContext.runFuncAsBillable(TaskContext.scala:285) at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.hasNext(PhotonBasicEvaluatorFactory.scala:289) at com.databricks.photon.CloseableIterator$$anon$10.hasNext(CloseableIterator.scala:211) at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$writeUsingPhoton$3(FileFormatWriter.scala:748) at org.apache.spark.scheduler.ResultTask.$anonfun$runTask$6(ResultTask.scala:107) at com.databricks.spark.util.ExecutorFrameProfiler$.record(ExecutorFrameProfiler.scala:110) at org.apache.spark.scheduler.ResultTask.$anonfun$runTask$1(ResultTask.scala:107) at com.databricks.spark.util.ExecutorFrameProfiler$.record(ExecutorFrameProfiler.scala:110) at org.apache.",2026-08-11T10:44:56.842Z
4fc616ca-cf1d-47e4-ba0e-1aebe0171bb0,silver,null,null,FAILED,"[CAST_INVALID_INPUT] The value '###CORRUPTED###' of the type ""STRING"" cannot be cast to ""DOUBLE"" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018 JVM stacktrace: org.apache.spark.SparkNumberFormatException at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:189) at com.databricks.photon.PhotonException$.getSparkException(PhotonException.scala:271) at com.databricks.photon.PhotonWriteResultHandler.getResult(PhotonWriteStageExec.scala:153) at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.open(PhotonBasicEvaluatorFactory.scala:264) at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.hasNextImpl(PhotonBasicEvaluatorFactory.scala:269) at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.$anonfun$hasNext$1(PhotonBasicEvaluatorFactory.scala:289) at scala.runtime.java8.JFunction0$mcZ$sp.apply(JFunction0$mcZ$sp.scala:17) at com.databricks.photon.metrics.BillableTimeTaskMetrics.withPhotonBilling(BillableTimeTaskMetrics.scala:71) at org.apache.spark.TaskContext.runFuncAsBillable(TaskContext.scala:285) at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.hasNext(PhotonBasicEvaluatorFactory.scala:289) at com.databricks.photon.CloseableIterator$$anon$10.hasNext(CloseableIterator.scala:211) at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$writeUsingPhoton$3(FileFormatWriter.scala:748) at org.apache.spark.scheduler.ResultTask.$anonfun$runTask$6(ResultTask.scala:107) at com.databricks.spark.util.ExecutorFrameProfiler$.record(ExecutorFrameProfiler.scala:11

In [0]:
%sql
SELECT transaction_id, price, typeof(price) AS price_type
FROM delta_forge_catalog.legacy_hms_db.bronze_transactions
WHERE file_name LIKE '%truncated_test%';

transaction_id,price,price_type
1,###CORRUPTED###,string
2,1.49,string
3,5.89,string


In [0]:
from pyspark.sql.functions import current_timestamp, lit, input_file_name,col
import uuid


run_id = str(uuid.uuid4())

def log_audit(layer, file_name, row_count, status, error_message=None):
    spark.sql(f"""
        INSERT INTO delta_forge_catalog.legacy_hms_db.pipeline_audit_log
        VALUES ('{run_id}', '{layer}', {f"'{file_name}'" if file_name else 'NULL'},
                {row_count if row_count is not None else 'NULL'}, '{status}',
                {f"'{error_message}'" if error_message else 'NULL'}, current_timestamp())
    """)

bronze_checkpoint = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/"
schema_location = "abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/"

log_audit("bronze", None, None, "STARTED")

try:
    bronze_df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .option("rescuedDataColumn", "_rescued_data")
        .load("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/")
        .withColumn("file_name", col("_metadata.file_path"))
        .withColumn("ingested_at", current_timestamp())
    )

    (bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", bronze_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
        .awaitTermination()
    )

    row_count = spark.table("delta_forge_catalog.legacy_hms_db.bronze_transactions").count()
    log_audit("bronze", None, row_count, "SUCCESS")
    print(f"Bronze SUCCESS — total rows: {row_count}")

except Exception as e:
    log_audit("bronze", None, None, "FAILED", str(e)[:500])
    raise

Bronze SUCCESS — total rows: 7100


In [0]:
dbutils.fs.mkdirs("/Volumes/delta-forge-catalog/bronze/raw_landing_vol/_quarantine/")

dbutils.fs.mv(
    "/Volumes/delta-forge-catalog/bronze/raw_landing_vol/2010-12-08.csv",
    "/Volumes/delta-forge-catalog/bronze/raw_landing_vol/_quarantine/2010-12-08.csv"
)
print("Moved incompatible file into subfolder — out of Auto Loader's ingestion scope.")

Moved incompatible file into subfolder — out of Auto Loader's ingestion scope.


In [0]:
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/bronze/", recurse=True)
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/checkpoints/schema/", recurse=True)
print("Bronze checkpoint and schema location cleared.")

Bronze checkpoint and schema location cleared.


In [0]:
log_audit(run_id, "bronze", None, None, "STARTED")

try:
    df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schema_location)
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
      .option("cloudFiles.rescuedDataColumn", "_rescued_data")
      .option("header", "true")
      .load(raw_path)
      .withColumn("file_name", col("_metadata.file_path"))
      .withColumn("file_arrival_time", col("_metadata.file_modification_time"))
    )

    query = (df.writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .option("mergeSchema", "true")
      .trigger(availableNow=True)
      .table("delta_forge_catalog.legacy_hms_db.bronze_transactions")
    )
    query.awaitTermination()

    row_count = spark.sql("SELECT COUNT(*) AS c FROM delta_forge_catalog.legacy_hms_db.bronze_transactions").collect()[0]["c"]
    log_audit(run_id, "bronze", None, row_count, "SUCCESS")
    print(f"Bronze SUCCESS — total rows: {row_count}")

except Exception as e:
    log_audit(run_id, "bronze", None, None, "FAILED", str(e))
    raise

Bronze SUCCESS — total rows: 8903


In [0]:
dbutils.fs.rm("abfss://delta-forge@deltaforge.dfs.core.windows.net/raw_landing/_quarantine/", recurse=True)

True